# Mushroom species classifier: Kaggle training

Trains `train.py` on a free Kaggle GPU and saves the best checkpoint plus a test report.

**Before you run it**

1. **Add the data:** *Add Input* → search for **combined-kaggle-mushrooms-dataset** (by *dariobaumberger*) → add it.
2. **Settings** (right-hand panel): *Accelerator* → **GPU T4 x2**, and *Internet* → **On**.
   Kaggle asks for phone verification the first time you turn on GPU or Internet.
3. **Code:** set `REPO_URL` below to your public GitHub repo, *or* upload the project folder as a
   private Kaggle dataset, add it as an input, and leave `REPO_URL` empty.
4. **Run:** for a quick check, set `EXTRA_ARGS = "--subset 20 --epochs 2"` and click *Run All*.
   For the real run, clear `EXTRA_ARGS`, then use **Save Version → Save & Run All (Commit)**.
   That runs in the background (up to 12 hours), so you can close the browser.

**Results** are in the saved version's **Output** tab: `checkpoints/best.pt` (use it with `predict.py`)
and `checkpoints/report/` (per-class CSV and confusion matrix).

**Cost:** Kaggle notebooks are free and need no payment details. GPU time counts against a weekly quota,
which is shown in the settings panel.

In [ ]:
# ---- settings ----
REPO_URL = ""            # e.g. "https://github.com/<your-username>/MushroomSpeciesPredictor.git"
ARCH = "efficientnet_b0" # or "resnet50" (slower, maybe more accurate), "resnet18", "convnet"
EPOCHS = 25
BATCH_SIZE = 64
EXTRA_ARGS = ""          # quick test: "--subset 20 --epochs 2"

In [ ]:
# ---- get the project code ----
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
CODE = f"{WORK}/code"
SPLIT = f"{WORK}/data_split"
OUT = f"{WORK}/checkpoints"

def find(pattern, max_depth=5):
    for depth in range(1, max_depth + 1):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * depth + pattern))
        if hits:
            return hits
    return []

if os.path.exists(CODE):
    shutil.rmtree(CODE)
if REPO_URL:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        raise SystemExit("git clone failed: check REPO_URL, that the repo is public, and that Internet is On.")
else:
    hits = find("train.py")
    if not hits:
        raise SystemExit("Set REPO_URL, or add the project code as an input dataset.")
    shutil.copytree(os.path.dirname(hits[0]), CODE)
print("Code files:", sorted(f for f in os.listdir(CODE) if f.endswith(".py")))

In [ ]:
# ---- find the dataset and split it 70/15/15 (symlinks, so nothing is copied) ----
candidates = [p for p in find("images") if os.path.isdir(p) and len(os.listdir(p)) > 50]
if not candidates:
    raise SystemExit("Dataset not found: add 'combined-kaggle-mushrooms-dataset' as an input.")
IMAGES = candidates[0]
print("Images:", IMAGES, "|", len(os.listdir(IMAGES)), "species")

# always rebuild: it takes seconds, and a half-finished split from an interrupted run would be wrong
shutil.rmtree(SPLIT, ignore_errors=True)
r = subprocess.run([sys.executable, "split_data.py", IMAGES, "--out", SPLIT, "--link"],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0:
    raise SystemExit("Splitting the dataset failed (see the message above).")

In [ ]:
# ---- check the GPU ----
import torch
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. In Settings set Accelerator to 'GPU T4 x2', or training will take days.")

In [ ]:
# ---- train (prints one line per epoch) ----
os.chdir(CODE)
!{sys.executable} train.py --data {SPLIT} --arch {ARCH} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers 4 --out {OUT} {EXTRA_ARGS}
if _exit_code != 0:
    raise SystemExit("Training failed: see the error above.")

In [ ]:
# ---- look at the results ----
import csv
from IPython.display import Image, display

with open(f"{OUT}/report/per_class.csv", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["accuracy"]]
rows.sort(key=lambda r: float(r["accuracy"]))
print("Weakest 15 species (accuracy, test images, most often confused with):")
for r in rows[:15]:
    print(f"  {float(r['accuracy']) * 100:5.1f}%  n={r['support']:>4}  {r['class']}  ->  {r['most_confused_with']}")
display(Image(f"{OUT}/report/confusion_matrix.png", width=900))

In [ ]:
# ---- clean up so the saved output is only the checkpoint and report ----
os.chdir(WORK)
shutil.rmtree(SPLIT, ignore_errors=True)   # removes only the symlinks, not the dataset
shutil.rmtree(CODE, ignore_errors=True)
for root, _, files in os.walk(OUT):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p) / 1e6:8.1f} MB  {os.path.relpath(p, WORK)}")